## 신경망 기초 이론과 PyTorch 구현

이번 파트에서는 신경망의 핵심 개념들을 이론적으로 이해하고, 이를 PyTorch로 직접 구현해보겠습니다.

**학습 목표:**

* 인공 신경망의 시초인 **퍼셉트론(Perceptron)**의 구조와 한계를 이해하고, 이를 극복하는 **다층 퍼셉트론(MLP)**의 원리를 설명할 수 있습니다.
* 신경망의 비선형성을 부여하는 다양한 **활성화 함수(Activation Function)**의 종류와 특징을 비교할 수 있습니다.
* 모델의 예측 성능을 측정하는 **손실 함수(Loss Function)**의 종류와 용도를 이해합니다.
* 신경망 학습의 핵심 알고리즘인 **역전파(Backpropagation)**의 과정을 설명하고 직접 구현할 수 있습니다.
* PyTorch를 사용해 간단한 **선형 회귀 모델**을 직접 구현하고 학습시켜 봅니다.

### 1. 퍼셉트론과 다층 퍼셉트론 개념 정리

퍼셉트론은 다수의 입력을 받아 하나의 출력을 내보내는 가장 단순한 형태의 인공 뉴런입니다.

**퍼셉트론의 수학적 표현:**
- 각 입력 신호($x_i$)에 고유한 **가중치($w_i$)**가 곱해집니다.
- 가중합($\sum_i w_i x_i$)에서 **편향(bias, $b$)**을 더한 값이 특정 **임계값**을 넘으면 1을, 그렇지 않으면 0을 출력합니다.

$$\hat y = f\left(\sum_i w_i x_i + b\right)$$

**단층 퍼셉트론의 한계:**
- 하나의 선으로 데이터를 나눌 수 없는 **선형 분리 불가능** 문제(예: XOR 게이트)는 해결할 수 없습니다.

**다층 퍼셉트론(MLP)의 해결책:**
- 입력층과 출력층 사이에 하나 이상의 **은닉층(Hidden Layer)**을 추가
- 은닉층을 통해 **비선형 변환**을 수행하여 복잡한 패턴 학습 가능
- **보편 근사 정리**: 충분히 많은 뉴런을 가진 은닉층이 하나만 있어도, MLP는 어떤 연속 함수든 원하는 정확도로 근사할 수 있습니다.


### 2. 활성화 함수와 손실 함수

#### 2.1 주요 활성화 함수들

활성화 함수는 선형 결합의 결과를 비선형 값으로 변환하여 모델의 표현력을 높이는 핵심 요소입니다.

| 함수 | 수식 | 장점 | 단점 | 주요 용도 |
|------|------|------|------|-----------|
| **Sigmoid** | $\sigma(z)=\frac{1}{1+e^{-z}}$ | 출력을 (0, 1) 사이 확률로 해석 가능 | 기울기 소실, Zero-centered가 아님 | 이진 분류 출력층 |
| **Tanh** | $\tanh(z) = \frac{e^z - e^{-z}}{e^z + e^{-z}}$ | 출력이 0 중심(Zero-centered) | 여전히 기울기 소실 문제 발생 | 과거 RNN의 은닉층 |
| **ReLU** | $\max(0, z)$ | 계산이 빠르고, 양수에서 기울기 소실 없음 | 음수에서 뉴런이 비활성화되는 "죽은 ReLU" 현상 | **(표준)** 대부분의 은닉층 |
| **Leaky ReLU** | $\max(\alpha z, z)$, $\alpha$는 작은 값 | "죽은 ReLU" 문제 완화 | $\alpha$ 값 선택이 추가 하이퍼파라미터 | ReLU의 대안 은닉층 |
| **Softmax** | $\frac{e^{z_j}}{\sum_k e^{z_k}}$ | 다중 클래스 출력을 확률 분포로 변환 | 입력(logit) 값이 크면 수치적으로 불안정 가능 | 다중 분류 출력층 |

#### 2.2 주요 손실 함수들

손실 함수는 모델의 예측값과 실제 정답 사이의 차이를 계산하는 함수입니다.

| 과제 유형 | 대표 손실 함수 | 핵심 아이디어 |
|-----------|----------------|---------------|
| **회귀 (Regression)** | **MSE** (Mean Squared Error) | 오차를 **제곱**하여 큰 오차에 더 큰 페널티를 부여. ($\frac{1}{n}\sum(y-\hat y)^2$) |
| (회귀 대안) | **MAE** (Mean Absolute Error) | 오차의 **절댓값**을 사용. 이상치(outlier)에 덜 민감함. ($\frac{1}{n}\sum|y-\hat y|$) |
| **이진 분류** | **Binary Cross-Entropy** | 정답 클래스에 대한 예측 확률을 최대화하는 방향으로 학습 (로그 우도 기반). |
| **다중 분류** | **Categorical Cross-Entropy** | Softmax 출력 확률 분포가 실제 정답(one-hot) 분포와 가까워지도록 학습. |

**참고:** 모델은 **손실 함수(예: MSE)를 최소화**하도록 학습되지만, 최종 성능은 사람이 해석하기 좋은 **측정 지표(예: R²)**로 평가하고 보고하는 경우가 많습니다.


### 3. 역전파 알고리즘 이해하기

역전파는 출력층에서 계산된 손실(오차)을 신경망의 뒤쪽에서 앞쪽으로 전파시키면서 각 가중치가 오차에 얼마나 기여했는지(기울기) 계산하고, 이를 이용해 가중치를 업데이트하는 과정입니다.

#### 역전파 4단계 요약

1. **순전파 (Forward Pass)**
   - 입력 데이터를 모델에 넣어 예측값을 계산하고, 최종적으로 스칼라 값의 **손실(Loss)**을 구합니다.

2. **지역 기울기 계산 (Compute Local Gradients)**
   - 각 노드의 출력값에 대한 손실 함수의 편미분 값을 계산합니다.

3. **역방향 전파 (Backward Pass)**
   - **연쇄 법칙(Chain Rule)**을 이용해 출력층부터 입력층까지 기울기를 역방향으로 전파하며, 각 가중치에 대한 손실의 편미분($\partial L / \partial w$)을 계산합니다.

4. **가중치 업데이트 (Update Weights)**
   - 계산된 기울기를 이용해 옵티마이저가 가중치를 업데이트합니다: $w \leftarrow w - \eta \frac{\partial L}{\partial w}$

#### 연쇄 법칙의 핵심

복합 함수의 미분을 단계별로 분해:

$$\frac{\partial E}{\partial w} = \frac{\partial E}{\partial \text{out}} \times \frac{\partial \text{out}}{\partial \text{net}} \times \frac{\partial \text{net}}{\partial w}$$

각 단계별로 계산하여 최종 기울기를 구합니다.

이제 실제로 간단한 신경망을 만들어 역전파 과정을 직접 구현해보겠습니다.


In [1]:
import torch
import numpy as np
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

# 시그모이드 활성화 함수와 그 미분
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(x):
    return x * (1 - x)

In [2]:
class SimpleNeuralNetwork:
    """
    간단한 2-2-1 신경망 (입력 2개, 은닉층 2개, 출력 1개)
    역전파를 직접 구현하여 XOR 문제를 해결해보겠습니다.
    """
    def __init__(self):
        # 가중치 초기화 (Xavier 초기화 사용)
        np.random.seed(42)
        
        # 입력층 -> 은닉층 가중치 (2x2)
        self.weights_input_hidden = np.random.uniform(-1, 1, (2, 2))
        
        # 은닉층 -> 출력층 가중치 (2x1)
        self.weights_hidden_output = np.random.uniform(-1, 1, (2, 1))
        
        # 편향 (bias)
        self.bias_hidden = np.random.uniform(-1, 1, (1, 2))
        self.bias_output = np.random.uniform(-1, 1, (1, 1))
        
        # 학습률
        self.learning_rate = 0.5
        
    def forward(self, X):
        """순전파"""
        # 입력층 -> 은닉층
        self.hidden_input = np.dot(X, self.weights_input_hidden) + self.bias_hidden
        self.hidden_output = sigmoid(self.hidden_input)
        
        # 은닉층 -> 출력층
        self.output_input = np.dot(self.hidden_output, self.weights_hidden_output) + self.bias_output
        self.predicted_output = sigmoid(self.output_input)
        
        return self.predicted_output
    
    def backward(self, X, y, predicted_output):
        """역전파 - 직접 구현"""
        # 출력층 오차 계산
        output_error = y - predicted_output
        output_delta = output_error * sigmoid_derivative(predicted_output)
        
        # 은닉층 오차 계산 (연쇄 법칙 적용)
        hidden_error = output_delta.dot(self.weights_hidden_output.T)
        hidden_delta = hidden_error * sigmoid_derivative(self.hidden_output)
        
        # 가중치와 편향 업데이트
        self.weights_hidden_output += self.hidden_output.T.dot(output_delta) * self.learning_rate
        self.bias_output += np.sum(output_delta, axis=0, keepdims=True) * self.learning_rate
        
        self.weights_input_hidden += X.T.dot(hidden_delta) * self.learning_rate
        self.bias_hidden += np.sum(hidden_delta, axis=0, keepdims=True) * self.learning_rate
    
    def train(self, X, y, epochs):
        """학습"""
        losses = []
        
        for epoch in range(epochs):
            # 순전파
            predicted_output = self.forward(X)
            
            # 손실 계산 (MSE)
            loss = np.mean((y - predicted_output) ** 2)
            losses.append(loss)
            
            # 역전파
            self.backward(X, y, predicted_output)
            
            if epoch % 1000 == 0:
                print(f"Epoch {epoch}, Loss: {loss:.6f}")
        
        return losses

In [3]:

# XOR 데이터셋 준비
X = np.array([[0, 0],
              [0, 1], 
              [1, 0],
              [1, 1]])

y = np.array([[0],
              [1],
              [1], 
              [0]])

In [4]:
print("XOR 데이터셋:")
print("입력:", X)
print("출력:", y.flatten())

XOR 데이터셋:
입력: [[0 0]
 [0 1]
 [1 0]
 [1 1]]
출력: [0 1 1 0]


In [5]:
# 신경망 생성 및 학습
nn = SimpleNeuralNetwork()
print("학습 시작...")
losses = nn.train(X, y, 10000)


학습 시작...
Epoch 0, Loss: 0.269134
Epoch 1000, Loss: 0.156267
Epoch 2000, Loss: 0.006921
Epoch 3000, Loss: 0.002990
Epoch 4000, Loss: 0.001870
Epoch 5000, Loss: 0.001351
Epoch 6000, Loss: 0.001054
Epoch 7000, Loss: 0.000862
Epoch 8000, Loss: 0.000728
Epoch 9000, Loss: 0.000630


In [6]:
# 학습 결과 확인
print("최종 예측 결과:")
final_predictions = nn.forward(X)
for i in range(len(X)):
    print(f"입력: {X[i]} -> 예측: {final_predictions[i][0]:.4f}, 실제: {y[i][0]}")

최종 예측 결과:
입력: [0 0] -> 예측: 0.0225, 실제: 0
입력: [0 1] -> 예측: 0.9747, 실제: 1
입력: [1 0] -> 예측: 0.9745, 실제: 1
입력: [1 1] -> 예측: 0.0205, 실제: 0


In [7]:
# 서브플롯 생성
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('학습 손실 변화', '학습 손실 변화 (상세)'),
    specs=[[{"secondary_y": False}, {"secondary_y": False}]]
)

# 첫 번째 그래프 - 전체 손실 변화
fig.add_trace(
    go.Scatter(x=list(range(len(losses))), y=losses, 
               mode='lines', name='전체 손실',
               line=dict(color='blue')),
    row=1, col=1
)

# 두 번째 그래프 - 상세 손실 변화 (100 에포크 이후)
fig.add_trace(
    go.Scatter(x=list(range(100, len(losses))), y=losses[100:], 
               mode='lines', name='상세 손실',
               line=dict(color='red')),
    row=1, col=2
)

# 레이아웃 업데이트
fig.update_layout(
    title_text="신경망 학습 손실 변화",
    showlegend=False,
    height=400
)

# x축과 y축 레이블 설정
fig.update_xaxes(title_text="Epoch", row=1, col=1)
fig.update_yaxes(title_text="Loss (MSE)", row=1, col=1)
fig.update_xaxes(title_text="Epoch", row=1, col=2)
fig.update_yaxes(title_text="Loss (MSE)", row=1, col=2)

# 격자 추가
fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')
fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor='lightgray')

fig.show()

In [8]:
print(f"최종 손실: {losses[-1]:.6f}")

# 학습된 가중치 확인
print("학습된 가중치:")
print("입력층 -> 은닉층 가중치:")
print(nn.weights_input_hidden)
print("은닉층 -> 출력층 가중치:")
print(nn.weights_hidden_output)
print("은닉층 편향:")
print(nn.bias_hidden)
print("출력층 편향:")
print(nn.bias_output)


최종 손실: 0.000555
학습된 가중치:
입력층 -> 은닉층 가중치:
[[-6.22927251  7.06823157]
 [ 6.48932803 -6.98440251]]
은닉층 -> 출력층 가중치:
[[-8.34651089]
 [-8.26054183]]
은닉층 편향:
[[3.13219759 3.57496502]]
출력층 편향:
[[12.26362826]]


#### 역전파 구현의 핵심 포인트

위의 구현에서 주목할 점들:

1. **연쇄 법칙의 적용**: 출력층의 오차를 은닉층으로 전파할 때 가중치를 곱해서 전달
2. **활성화 함수의 미분**: 시그모이드 함수의 미분을 사용하여 기울기 계산
3. **가중치 업데이트**: 계산된 기울기에 학습률을 곱해서 가중치 조정

이제 같은 문제를 PyTorch의 자동 미분 기능을 사용해서 해결해보겠습니다.

### 4. PyTorch로 선형 회귀 모델 구현하기

이제 실제 데이터를 사용하여 선형 회귀 모델을 구현해보겠습니다. California 주택 가격 데이터셋을 사용하여 8개의 주택 관련 특성으로 주택 가격을 예측하는 모델을 만들어보겠습니다.

#### 4.1 데이터 준비 및 전처리


In [9]:
# 필요한 라이브러리 임포트
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader
import numpy as np

# 1. 데이터 로드 및 탐색
print("California 주택 가격 데이터셋 로드 중...")
housing = fetch_california_housing()
X, y = housing.data, housing.target

print(f"데이터 형태: {X.shape}")
print(f"타겟 형태: {y.shape}")
print(f"특성 이름: {housing.feature_names}")
print(f"타겟 설명: 주택 중간 가격 (단위: $100,000)")

# 데이터 기본 통계
print(f"특성 데이터 통계:")
print(f"평균: {X.mean(axis=0)}")
print(f"표준편차: {X.std(axis=0)}")


California 주택 가격 데이터셋 로드 중...
데이터 형태: (20640, 8)
타겟 형태: (20640,)
특성 이름: ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
타겟 설명: 주택 중간 가격 (단위: $100,000)
특성 데이터 통계:
평균: [ 3.87067100e+00  2.86394864e+01  5.42899974e+00  1.09667515e+00
  1.42547674e+03  3.07065516e+00  3.56318614e+01 -1.19569704e+02]
표준편차: [1.89977569e+00 1.25852527e+01 2.47411320e+00 4.73899376e-01
 1.13243469e+03 1.03857980e+01 2.13590065e+00 2.00348319e+00]


In [10]:
print(f"타겟 데이터 통계:")
print(f"최소값: {y.min():.2f}, 최대값: {y.max():.2f}, 평균: {y.mean():.2f}")

타겟 데이터 통계:
최소값: 0.15, 최대값: 5.00, 평균: 2.07


In [11]:
# 2. 데이터 분할
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"훈련 데이터: {X_train.shape}, 검증 데이터: {X_val.shape}")

훈련 데이터: (16512, 8), 검증 데이터: (4128, 8)


In [12]:
# 3. 데이터 스케일링 (표준화)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

print("데이터 스케일링 완료!")
print(f"스케일링 후 훈련 데이터 평균: {X_train_scaled.mean(axis=0)}")
print(f"스케일링 후 훈련 데이터 표준편차: {X_train_scaled.std(axis=0)}")


데이터 스케일링 완료!
스케일링 후 훈련 데이터 평균: [-6.59266865e-15 -6.68608149e-17  8.01559239e-15 -1.17273358e-15
 -2.60880895e-18 -1.13675656e-16  7.99652724e-14 -3.87910056e-13]
스케일링 후 훈련 데이터 표준편차: [1. 1. 1. 1. 1. 1. 1. 1.]


In [13]:
# 4. PyTorch Tensor로 변환 및 Dataset, DataLoader 생성
train_ds = TensorDataset(
    torch.tensor(X_train_scaled, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)  # (n,) -> (n,1) 형태로 변환
)

val_ds = TensorDataset(
    torch.tensor(X_val_scaled, dtype=torch.float32),
    torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
)

In [14]:
# DataLoader 생성
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=256)

print(f"훈련 DataLoader 배치 수: {len(train_dl)}")
print(f"검증 DataLoader 배치 수: {len(val_dl)}")

훈련 DataLoader 배치 수: 258
검증 DataLoader 배치 수: 17


In [15]:
# 첫 번째 배치 확인
features_batch, labels_batch = next(iter(train_dl))
print(f"배치 특성 형태: {features_batch.shape}")
print(f"배치 라벨 형태: {labels_batch.shape}")


배치 특성 형태: torch.Size([64, 8])
배치 라벨 형태: torch.Size([64, 1])


#### 4.2 선형 회귀 모델 정의

간단한 선형 회귀 모델부터 시작해서, 점진적으로 복잡한 모델로 발전시켜보겠습니다.

In [16]:
# 1. 단순 선형 회귀 모델 (Linear Layer 하나만)
class SimpleLinearRegression(nn.Module):
    def __init__(self, input_features):
        super(SimpleLinearRegression, self).__init__()
        self.linear = nn.Linear(input_features, 1)
    
    def forward(self, x):
        return self.linear(x)

In [17]:
# 2. 다층 퍼셉트론 회귀 모델 (은닉층 추가)
class MLPRegression(nn.Module):
    def __init__(self, input_features, hidden_size=32):
        super(MLPRegression, self).__init__()
        self.hidden1 = nn.Linear(input_features, hidden_size)
        self.hidden2 = nn.Linear(hidden_size, hidden_size // 2)
        self.output = nn.Linear(hidden_size // 2, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)  # 과적합 방지
    
    def forward(self, x):
        x = self.relu(self.hidden1(x))
        x = self.dropout(x)
        x = self.relu(self.hidden2(x))
        x = self.dropout(x)
        x = self.output(x)
        return x

In [18]:
# 모델 인스턴스 생성
input_features = X_train.shape[1]
print(f"입력 특성 수: {input_features}")

# 두 모델 모두 생성
simple_model = SimpleLinearRegression(input_features)
mlp_model = MLPRegression(input_features, hidden_size=64)

print("단순 선형 회귀 모델:")
print(simple_model)
print("다층 퍼셉트론 회귀 모델:")
print(mlp_model)

입력 특성 수: 8
단순 선형 회귀 모델:
SimpleLinearRegression(
  (linear): Linear(in_features=8, out_features=1, bias=True)
)
다층 퍼셉트론 회귀 모델:
MLPRegression(
  (hidden1): Linear(in_features=8, out_features=64, bias=True)
  (hidden2): Linear(in_features=64, out_features=32, bias=True)
  (output): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.2, inplace=False)
)


In [19]:
# 모델 파라미터 수 계산
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"단순 선형 회귀 모델 파라미터 수: {count_parameters(simple_model)}")
print(f"다층 퍼셉트론 모델 파라미터 수: {count_parameters(mlp_model)}")


단순 선형 회귀 모델 파라미터 수: 9
다층 퍼셉트론 모델 파라미터 수: 2689


#### 4.3 모델 학습 함수 정의

두 모델을 비교하기 위해 학습과 평가를 위한 함수들을 정의하겠습니다.

In [20]:
def train_model(model, train_loader, val_loader, num_epochs=100, learning_rate=1e-3):
    """
    모델 학습 함수
    """
    # 손실 함수와 옵티마이저 정의
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    # 학습 기록을 위한 리스트
    train_losses = []
    val_losses = []
    
    print(f"학습 시작 - 에포크: {num_epochs}, 학습률: {learning_rate}")
    
    for epoch in range(num_epochs):
        # 훈련 모드
        model.train()
        train_loss = 0.0
        
        for batch_features, batch_labels in train_loader:
            # 기울기 초기화
            optimizer.zero_grad()
            
            # 순전파
            predictions = model(batch_features)
            loss = criterion(predictions, batch_labels)
            
            # 역전파
            loss.backward()
            
            # 가중치 업데이트
            optimizer.step()
            
            train_loss += loss.item()
        
        # 평균 훈련 손실
        avg_train_loss = train_loss / len(train_loader)
        train_losses.append(avg_train_loss)
        
        # 검증 모드
        model.eval()
        val_loss = 0.0
        
        with torch.no_grad():
            for batch_features, batch_labels in val_loader:
                predictions = model(batch_features)
                loss = criterion(predictions, batch_labels)
                val_loss += loss.item()
        
        # 평균 검증 손실
        avg_val_loss = val_loss / len(val_loader)
        val_losses.append(avg_val_loss)
        
        # 10 에포크마다 출력
        if (epoch + 1) % 10 == 0:
            print(f'Epoch [{epoch+1:3d}/{num_epochs}] | '
                  f'Train Loss: {avg_train_loss:.4f} | '
                  f'Val Loss: {avg_val_loss:.4f}')
    
    return train_losses, val_losses

In [21]:
def evaluate_model(model, val_loader):
    """
    모델 평가 함수 (R² 스코어 계산)
    """
    model.eval()
    all_predictions = []
    all_targets = []
    
    with torch.no_grad():
        for batch_features, batch_labels in val_loader:
            predictions = model(batch_features)
            all_predictions.extend(predictions.cpu().numpy())
            all_targets.extend(batch_labels.cpu().numpy())
    
    all_predictions = np.array(all_predictions)
    all_targets = np.array(all_targets)
    
    # R² 스코어 계산
    ss_res = np.sum((all_targets - all_predictions) ** 2)
    ss_tot = np.sum((all_targets - np.mean(all_targets)) ** 2)
    r2_score = 1 - (ss_res / ss_tot)
    
    # RMSE 계산
    rmse = np.sqrt(np.mean((all_targets - all_predictions) ** 2))
    
    return r2_score, rmse, all_predictions, all_targets

#### 4.4 모델 학습 및 비교

In [22]:
# 3. 모델 평가
print("\n" + "="*50)
print("모델 평가 결과")
print("="*50)

# 단순 선형 회귀 모델 평가
simple_r2, simple_rmse, simple_pred, simple_target = evaluate_model(simple_model, val_dl)
print(f"단순 선형 회귀 모델:")
print(f"  R² Score: {simple_r2:.4f}")
print(f"  RMSE: {simple_rmse:.4f}")

# 다층 퍼셉트론 모델 평가
mlp_r2, mlp_rmse, mlp_pred, mlp_target = evaluate_model(mlp_model, val_dl)
print(f"\n다층 퍼셉트론 모델:")
print(f"  R² Score: {mlp_r2:.4f}")
print(f"  RMSE: {mlp_rmse:.4f}")

print(f"\n성능 개선:")
print(f"  R² Score 개선: {mlp_r2 - simple_r2:.4f}")
print(f"  RMSE 개선: {simple_rmse - mlp_rmse:.4f}")



모델 평가 결과
단순 선형 회귀 모델:
  R² Score: -2.8708
  RMSE: 2.2522

다층 퍼셉트론 모델:
  R² Score: -2.7443
  RMSE: 2.2151

성능 개선:
  R² Score 개선: 0.1265
  RMSE 개선: 0.0371


In [23]:
# 학습된 가중치 분석 (단순 선형 회귀)
print("="*50)
print("학습된 가중치 분석 (단순 선형 회귀)")
print("="*50)

weights = simple_model.linear.weight.data.numpy().flatten()
bias = simple_model.linear.bias.data.numpy()[0]

print("특성별 가중치:")
for i, (feature_name, weight) in enumerate(zip(housing.feature_names, weights)):
    print(f"  {feature_name:15s}: {weight:8.4f}")
print(f"  {'Bias':15s}: {bias:8.4f}")


학습된 가중치 분석 (단순 선형 회귀)
특성별 가중치:
  MedInc         :   0.1611
  HouseAge       :   0.1207
  AveRooms       :   0.1940
  AveBedrms      :   0.2756
  Population     :   0.2570
  AveOccup       :   0.1626
  Latitude       :   0.0179
  Longitude      :  -0.0803
  Bias           :   0.1621


In [24]:
# 가중치 시각화
fig = go.Figure(data=[
    go.Bar(x=housing.feature_names, y=weights)
])

fig.update_layout(
    title='특성별 가중치 (단순 선형 회귀)',
    xaxis_title='특성',
    yaxis_title='가중치',
    xaxis=dict(tickangle=45),
    showlegend=False,
    width=800,
    height=500
)

fig.show()

### 5. 정리 및 핵심 포인트

이번 실습을 통해 다음과 같은 내용을 학습했습니다:

#### 5.1 역전파 알고리즘의 이해
- **직접 구현**: NumPy를 사용하여 역전파 과정을 단계별로 구현
- **연쇄 법칙**: 복합 함수의 미분을 단계별로 분해하여 계산
- **PyTorch 자동 미분**: `loss.backward()`를 통한 자동 기울기 계산

#### 5.2 선형 회귀 모델 구현
- **데이터 전처리**: 표준화(StandardScaler) 적용의 중요성
- **모델 비교**: 단순 선형 회귀 vs 다층 퍼셉트론의 성능 차이
- **평가 지표**: MSE, RMSE, R² 스코어를 통한 모델 성능 평가

#### 5.3 PyTorch 핵심 구성 요소
- **Dataset & DataLoader**: 효율적인 데이터 처리 파이프라인
- **nn.Module**: 모델 정의를 위한 기본 클래스
- **Optimizer**: Adam, SGD 등 다양한 최적화 알고리즘
- **Loss Functions**: MSE, Cross-Entropy 등 목적에 맞는 손실 함수 선택

#### 5.4 실무에서의 팁
1. **데이터 스케일링**: 특성들의 범위가 다를 때 반드시 수행
2. **모델 복잡도**: 단순한 모델부터 시작하여 점진적으로 복잡도 증가
3. **과적합 방지**: Dropout, 정규화 등의 기법 활용
4. **성능 모니터링**: 훈련/검증 손실을 함께 관찰하여 과적합 여부 확인

#### 5.5 다음 단계
- **더 복잡한 모델**: CNN, RNN 등 다양한 신경망 구조 학습
- **하이퍼파라미터 튜닝**: 학습률, 배치 크기, 모델 구조 최적화
- **고급 기법**: 배치 정규화, 잔차 연결, 어텐션 메커니즘 등

이제 신경망의 기본 이론과 PyTorch 구현 방법을 모두 익혔습니다. 다음 파트에서는 더 복잡한 문제들을 해결해보겠습니다!
